In [14]:
# Blibiotecas
import os
from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

In [15]:
load_dotenv()

DB_PATH = "../data/vectorstore"

# Recarrega o mesmo modelo de embeddings open-soure da Sprint 1
embeddings = HuggingFaceBgeEmbeddings(model_name="all-MiniLM-L6-v2")

# Conecta-se ao banco vetorial existente
vectorstore = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embeddings
)

# Cria o recuperador de contexto (Retriever) buscando os 3 trechos mais similares
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Banco vetorial conectado com sucesso!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 689.81it/s]


Banco vetorial conectado com sucesso!


In [16]:
PROMPT_TEMPLATE = """
Você é um auditor especialista em análise de contratos corporativos.
Sua missão é responder à pergunta do usuário baseando-se EXCLUSIVAMENTE nos trechos de contrato fornecidos no Contexto abaixo.

REGRAS DE COMPLIANCE ESTREITAS:
1. Se a resposta não estiver explicitamente mencionada no Contexto abaixo, responda EXATAMENTE: "Informação não encontrada na documentação fornecida." Não invente ou presuma nada.
2. Para cada afirmação ou dado citado na sua resposta, você DEVE apontar a fonte no final da frase no seguinte formato: [Fonte: NOME_DO_ARQUIVO | Pág: X].
3. Seja direto, analítico e imparcial.

Contexto Recuperado do Banco Vetorial:
{context}

---
Pergunta do Usuário: {question}

Resposta Analítica com Citação de Fontes:
"""

prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
print("Template de Prompt Anti-Alucinação configurado!")

Template de Prompt Anti-Alucinação configurado!


In [19]:
def query_contract_copilot(user_query: str) -> dict:
    """
    Executa a busca vetorial no ChromaDB e gera uma resposta fundamentada
    com citação de fontes utilizando o LLM
    """
    # 1. Recupera os documentos relevantes do banco vetorial
    retrieved_docs = retriever.invoke(user_query)

    # 2. Formata o contexto injetando metadados de rastreabilidade (Arquivo e Página)
    context_parts = []
    for doc in retrieved_docs:
        source_file = os.path.basename(doc.metadata.get("source", "contrato_desconhecido.pdf"))
        # Páginas no PyPDF começam no índice 0, somamos 1 para exibição humana
        page_num = doc.metadata.get("page", 0) + 1
        context_parts.append(f"[Arquivo: {source_file} | Pág: {page_num}]\n{doc.page_content}")

    formatted_context = "\n\n---\n\n".join(context_parts)

    # 3. Configura o modelo (temperature=0 garante determinismo e vite alucinações)
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    # 4. Monta e executa a cadeia (LCEL - LangChain Expression Language)
    chain = prompt_template | llm | StrOutputParser()

    answer = chain.invoke({
        "context": formatted_context,
        "question": user_query
    })

    return {
        "pergunta": user_query,
        "resposta": answer,
        "documentos_consultados": [os.path.basename(doc.metadata.get("source", "")) for doc in retrieved_docs]   
    }

print("Função query_contract_copilot criada com sucesso!")


Função query_contract_copilot criada com sucesso!


In [20]:
# TESTE 1: Informação existente no Contrato 01
pergunta_1 = "Qual é a porcentagem da multa rescisória estipulada no contrato de serviços de TI e qual o índice de reajuste?"
resultado_1 = query_contract_copilot(pergunta_1)

print("=== TESTE 1: Consulta Válida ===")
print("Pergunta:", resultado_1["pergunta"])
print("\nResposta:", resultado_1["resposta"])
print("\nFontes Consultadas:", set(resultado_1["documentos_consultados"]))

print("\n" + "="*50 + "\n")

# TESTE 2: Informação inexistente (Teste Anti-Alucinação / Zero Alucinação)
pergunta_2 = "Qual é o valor diário do vale-refeição e do auxílio-creche mencionado no contrato?"
resultado_2 = query_contract_copilot(pergunta_2)

print("=== TESTE 2: Teste Anti-Alucinação ===")
print("Pergunta:", resultado_2["pergunta"])
print("\nResposta:", resultado_2["resposta"])

=== TESTE 1: Consulta Válida ===
Pergunta: Qual é a porcentagem da multa rescisória estipulada no contrato de serviços de TI e qual o índice de reajuste?

Resposta: A porcentagem da multa rescisória estipulada no contrato de serviços de TI é de 15% sobre o valor total do contrato [Fonte: Prestação de Serviços de TI (Cenário de Alto Risco).pdf | Pág: 1].

O índice de reajuste utilizado no contrato de serviços de TI é o IPCA acumulado no período [Fonte: Prestação de Serviços de TI (Cenário de Alto Risco).pdf | Pág: 1].

Já no contrato de consultoria de marketing, não há índice pré-fixado estipulado para correções automáticas [Fonte: Consultoria de Marketing (Cenário de Atenção).pdf | Pág: 1].

Fontes Consultadas: {'Consultoria de Marketing (Cenário de Atenção).pdf', 'Prestação de Serviços de TI (Cenário de Alto Risco).pdf'}


=== TESTE 2: Teste Anti-Alucinação ===
Pergunta: Qual é o valor diário do vale-refeição e do auxílio-creche mencionado no contrato?

Resposta: Informação não encont